In [ ]:
#base environment for python scripts

In [ ]:
import sys
import os
import argparse
import numpy as np
import pandas as pd

from collections import defaultdict
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

In [ ]:
#global variables

In [ ]:
NARROWPEAK_SCHEMA = ["chr", "start", "end", "1", "2", "3", "4", "5", "6", "summit"]

In [ ]:
#load peaks and hits

In [ ]:
vista_common_name = pd.read_csv("~/chrombpnet/chromnpnet_cardiomyocytes/peak_from_ArchR_1000bp/data/region_for_predict/vista/heart/liftover/heart_tissue_VISTA_common_name.txt",header=None,names=["name"])
vista_common_name

In [ ]:
vista_common_name_list=vista_common_name["name"].to_list()
vista_common_name_list

In [ ]:
X_peaks_coords = pd.read_csv("~/chrombpnet/chromnpnet_cardiomyocytes/peak_from_ArchR_1000bp/data/region_for_predict/vista/heart/heart_tissue_VISTA.10col.bed", sep='\t', header=None, names = NARROWPEAK_SCHEMA)
X_peaks_coords.head()

In [ ]:
X_peaks_coords=X_peaks_coords[X_peaks_coords["1"].isin(vista_common_name_list)]

In [ ]:
X_peaks_coords.shape

In [ ]:
max(X_peaks_coords.index)

In [ ]:
len(X_peaks_coords.index)

In [ ]:
hits = pd.read_csv("~/chrombpnet/chromnpnet_cardiomyocytes/peak_from_ArchR_1000bp/chrombpnet_model_predicting/finemo/heart_tissue_VISTA/counts/hit_call/hits.tsv", sep="\t")
hits.head()

In [ ]:
hits["original_peak_name"]=hits["peak_name"].str.split("_").str[0]
hits

In [ ]:
hits = hits[hits['original_peak_name'].isin(X_peaks_coords["1"])]
len(hits)

In [ ]:
#hits=hits[hits["hit_coefficient"]>10]
#hits

In [ ]:
#hits = hits.merge(motif_annotation[['motif_name', 'category', 'annotation_broad']], on=['motif_name'], how='left')

In [ ]:
#Find motifs that significantly co-occur with the motif of interest using Fisher's exact test.

In [ ]:
# for each peak, get the set of unique motifs that occur in it
peak_to_motifs = hits.groupby('original_peak_name')['motif_name'].unique().apply(list)

In [ ]:
# create a mapping from a motif to all the peaks it's in
motif_to_peaks = defaultdict(set)
for original_peak_name, motifs in peak_to_motifs.items():
    for motif in motifs:
        motif_to_peaks[motif].add(original_peak_name)

In [ ]:
all_motif_names = motif_to_peaks.keys()	
all_peak_names = hits['original_peak_name'].unique().tolist()

In [ ]:
def find_cooccurring_motifs(hits, motif_of_interest, correct = False):
	"""
	Find motifs that significantly co-occur with the motif of interest using Fisher's exact test.
	"""	

	# for each peak, get the set of unique motifs that occur in it
	peak_to_motifs = hits.groupby('original_peak_name')['motif_name'].unique().apply(list)

	# create a mapping from a motif to all the peaks it's in
	motif_to_peaks = defaultdict(set)
	for original_peak_name, motifs in peak_to_motifs.items():
		for motif in motifs:
			motif_to_peaks[motif].add(original_peak_name)

	all_motif_names = motif_to_peaks.keys()	
	all_peak_ids = hits['original_peak_name'].unique().tolist()
	
	results = []

	for partner_motif in all_motif_names:

		if partner_motif == motif_of_interest:
			continue

		# drawing peaks with instances of one motif, and estimating the probability that 
		# we see instances of another, modelled by the hypergeometric distribution:
		# N: total number of peaks w/ motif instances
		# n: number of peaks w/ motif of interest
		# K: total number of peaks w/ other motif
		# k: number of peaks with both motifs
		N = len(all_peak_ids)

		peaks_with_interest = motif_to_peaks.get(motif_of_interest, set())
		n = len(peaks_with_interest)

		peaks_with_partner = motif_to_peaks.get(partner_motif, set())
		K = len(peaks_with_partner)

		peaks_with_both = peaks_with_interest.intersection(peaks_with_partner)
		k = len(peaks_with_both)

		if k == 0 or n == 0 or K == 0:
			continue

		contingency_table = [[k, n - k], [K - k, N - n - K + k]]

		# 'greater' tests the hypothesis that the odds ratio is > 1 (enrichment)
		# the odds ratio is the ratio of:
		# - the odds of motif B occurring in peaks with motif A
		# - the odds of motif B occurring in peaks without motif A
		odds_ratio, p_value = fisher_exact(contingency_table, alternative='greater')

		results.append({
			'motif_A': motif_of_interest,
			#'class_A': hits[hits['motif_name'] == motif_of_interest]['pattern_class'].iloc[0],
			'motif_B': partner_motif,
			#'class_B': hits[hits['motif_name'] == partner_motif]['pattern_class'].iloc[0],
			'p_value': p_value,
			'odds_ratio': odds_ratio,
			'num_A': n,
			'num_B': K,
			'num_both': k
		})

	results_df = pd.DataFrame(results)

	if correct:
		p_values = results_df['p_value']
		reject, q_values, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')
		results_df['q_value'] = q_values
		results_df['is_significant'] = reject
		co_motifs = results_df[results_df['is_significant']].sort_values(by='odds_ratio', ascending=False)
		print(f"Top co-occurring motifs for {motif_of_interest}:")
		return co_motifs
	else:
		return results_df

In [ ]:
#Loop over all base motifs and run the co-occurrence enrichment analysis using Fisher's exact test.

In [ ]:
out_path = "~/chrombpnet/chromnpnet_cardiomyocytes/peak_from_ArchR_1000bp/chrombpnet_model_predicting/finemo/heart_tissue_VISTA/counts/Hs_C6.2_heart_tissue_VISTA_cooccurrence_fisher_exact.tsv"

In [ ]:
results_all = []
# loop over each base motif (broad category)
for motif in all_motif_names:
    print(motif)
    res = find_cooccurring_motifs(hits, motif, correct=False)
    results_all.append(res)

results_all_df = pd.concat(results_all, ignore_index=True)

# multiple testing correction
p_values = results_all_df['p_value']
reject, q_values, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')
#reject, q_values, _, _ = multipletests(p_values, alpha=0.5, method='fdr_bh')
results_all_df['q_value'] = q_values
results_all_df['is_significant'] = reject

# significant results
cooc_motifs = results_all_df[results_all_df['is_significant']].sort_values(by='odds_ratio', ascending=False)

cooc_motifs['log10q'] = -np.log10(cooc_motifs['q_value'] + 1e-300)  # add small value to avoid log(0)
cooc_motifs['log2odds'] = np.log2(cooc_motifs['odds_ratio'])

# assign cluster
#cooc_motifs['cluster'] = cluster

print(f"原始测试对总数: {len(results_all_df)}")
if len(results_all_df) > 0:
    print(f"显著性结果总数 (q < 0.5): {len(cooc_motifs)}")
    print("Q值前五位:", results_all_df['q_value'].sort_values().head())
else:
    print("错误：results_all_df 是空的！请检查 find_cooccurring_motifs 函数。")

# write to TSV
print(f"@ writing results to {out_path}")
cooc_motifs.to_csv(out_path, sep="\t", index=False)

In [ ]:
motif_matrix = pd.crosstab(hits['original_peak_name'], hits['motif_name'])
motif_matrix.head()

In [ ]:
column_sums = motif_matrix.sum()
column_sums
# 按总和降序排序
sorted_columns = column_sums.sort_values(ascending=False)
sorted_columns
top_6_columns = sorted_columns.head(6).index.tolist()
top_6_columns

In [ ]:
motif_matrix.to_csv("~/chrombpnet/chromnpnet_cardiomyocytes/peak_from_ArchR_1000bp/chrombpnet_model_predicting/finemo/heart_tissue_VISTA/counts/Hs_C6.2_heart_tissue_VISTA_motif_presence_matrix.csv")

In [ ]:
#Create Binary Matrix for Co-occurrence (Present or Not)二值矩阵
motif_binary = (motif_matrix > 0).astype(int)

In [ ]:
# Calculate Co-occurrence Matrix共现矩阵
# (Transpose) dot (Matrix) -> (Motifs x Peaks) dot (Peaks x Motifs) -> (Motifs x Motifs)
cooccurrence_matrix = motif_binary.T.dot(motif_binary)

In [ ]:
cooccurrence_matrix.to_csv("~/chrombpnet/chromnpnet_cardiomyocytes/peak_from_ArchR_1000bp/chrombpnet_model_predicting/finemo/heart_tissue_VISTA/counts/Hs_C6.2_heart_tissue_VISTA_motif_cooccurrence_matrix.csv")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
#Plot Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(cooccurrence_matrix, annot=True, fmt='d', cmap='Reds')
plt.title('Motif Co-occurrence Matrix (Number of Peaks shared)')
#plt.savefig('motif_cooccurrence.png')
plt.show()

In [ ]:
# --- 计算余弦相似度 ---

# 计算每个 Motif 出现的总次数 (得到对角线元素)
# 这相当于每个向量的模的平方 (|A|^2)
motif_sums = np.diag(cooccurrence_matrix) 

# 计算分母矩阵 (|A| * |B|)
# 利用外积 (outer product) 计算所有两两组合的模之积
# sqrt(count_A) * sqrt(count_B)
denom = np.outer(np.sqrt(motif_sums), np.sqrt(motif_sums))

# 计算余弦相似度矩阵
# 避免分母为 0 的情况（如果有 Motif 一次都没出现）
# 用 with np.errstate 忽略除以 0 的警告，或者直接处理
cosine_sim_matrix = cooccurrence_matrix / denom

# 将 NaN (除以0产生) 替换为 0
cosine_sim_matrix = cosine_sim_matrix.fillna(0) # 如果是 DataFrame
# 如果是 numpy array: cosine_sim_matrix[np.isnan(cosine_sim_matrix)] = 0

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(cosine_sim_matrix, 
            annot=False, # 相似度通常数值较多，建议关掉数值或只显示一部分
            cmap='viridis', # 或 'coolwarm', 'Reds'
            vmin=0, vmax=1) # 余弦相似度范围通常是 0 到 1 (对于非负数据)
plt.title('Motif Co-occurrence (Cosine Similarity)')
plt.show()